In [1]:
import uproot
file=uproot.open(r"C:\Users\Agastya\Hbb-jet-tagging\data\raw\train_rooot_file.root")
file.keys()
tree=file["deepntuplizer/tree;41"]
tree.keys()
tree.show()
import awkward as ak
import pandas as pd
import numpy as np

feature_cols = [
    
    "fj_pt",
    "fj_sdmass",    
    "fj_sdn2",
    "fj_n_sdsubjets",
    "fj_tau1",
    "fj_tau2",
    "fj_tau3",
    "fj_ptDR",
    "fj_relptdiff",
    "fj_jetNTracks",
    "fj_nSV",
    "fj_sdsj1_pt",
    "fj_sdsj1_mult",
    "fj_sdsj1_ptD",
    "fj_mass",
    "fj_eta",
    "fj_phi",
    "fj_sdsj1_eta",
    "fj_sdsj1_mass"
]
label_col = "label_H_bb"

arrays = tree.arrays(feature_cols + [label_col], library="np")
df=pd.DataFrame(arrays)
df.head()
df.insert(
    loc=7,
    column="tau21",
    value=df["fj_tau2"]/df["fj_tau1"]
)
df.insert(
    loc=8,
    column="tau32",
    value=df["fj_tau3"]/df["fj_tau2"]
)
df.head()






name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
Delta_gen_pt         | float                    | AsDtype('>f4')
event_no             | uint32_t                 | AsDtype('>u4')
gen_pt               | float                    | AsDtype('>f4')
isB                  | int32_t                  | AsDtype('>i4')
isBB                 | int32_t                  | AsDtype('>i4')
isC                  | int32_t                  | AsDtype('>i4')
isG                  | int32_t                  | AsDtype('>i4')
isLeptonicB          | int32_t                  | AsDtype('>i4')
isLeptonicB_C        | int32_t                  | AsDtype('>i4')
isS                  | int32_t                  | AsDtype('>i4')
isUD                 | int32_t                  | AsDtype('>i4')
isUndefined          | int32_t                  | AsDtype('>i4')
jet_corr_pt          | float                    | AsDtype(

,fj_pt,fj_sdmass,fj_sdn2,fj_n_sdsubjets,fj_tau1,fj_tau2,fj_tau3,tau21,tau32,fj_ptDR,...,fj_nSV,fj_sdsj1_pt,fj_sdsj1_mult,fj_sdsj1_ptD,fj_mass,fj_eta,fj_phi,fj_sdsj1_eta,fj_sdsj1_mass,label_H_bb
0,448.633636,4.911332,0.000073,2.0,0.067988,0.059589,0.044930,0.876467,0.753994,11.378714,...,0.0,349.067657,6.0,0.525917,68.060570,-0.277724,-2.183877,-0.280709,3.302880,0
1,868.861389,34.992409,0.001675,2.0,0.111101,0.087648,0.072055,0.788904,0.822090,112.677155,...,0.0,699.895508,19.0,0.412773,161.474731,-0.716812,2.365500,-0.708606,19.311876,0
2,573.674377,13.481631,0.000554,2.0,0.073106,0.061262,0.049685,0.837985,0.811027,27.063063,...,1.0,404.355164,9.0,0.682750,86.054962,0.219193,0.296306,0.219143,6.023303,0
3,206.318527,18.484650,0.005919,2.0,0.132059,0.086257,0.064672,0.653168,0.749767,42.051800,...,2.0,176.024475,13.0,0.331445,35.347668,0.336462,2.244762,0.337573,11.302593,0
4,1068.199097,99.316536,0.012706,2.0,0.116926,0.040401,0.032547,0.345525,0.805614,195.188263,...,1.0,696.555420,19.0,0.360593,139.384796,0.889782,-2.385019,0.917607,22.814959,1


In [2]:
from scipy.stats import ks_2samp

ks_results = []

for col in df.columns:
    if col == label_col:
        continue

    sig = df.loc[df[label_col] == 1, col]
    bkg = df.loc[df[label_col] == 0, col]

    ks, _ = ks_2samp(sig, bkg)
    ks_results.append((col, ks))

ks_df = (
    pd.DataFrame(ks_results, columns=["feature", "ks"])
    .sort_values("ks", ascending=False)
)

ks_df


,feature,ks
1,fj_sdmass,0.606585
9,fj_ptDR,0.600912
7,tau21,0.545917
2,fj_sdn2,0.463712
16,fj_mass,0.459075
20,fj_sdsj1_mass,0.443128
14,fj_sdsj1_mult,0.335394
12,fj_nSV,0.330393
0,fj_pt,0.230287
13,fj_sdsj1_pt,0.223018


In [4]:
import uproot
import pandas as pd

TRAIN_ROOT = r"C:\Users\Agastya\Hbb-jet-tagging\data\raw\train_rooot_file.root"
TREE_NAME = "deepntuplizer/tree;41"

feature_cols = [
    "fj_pt", "fj_sdmass", "fj_sdn2", "fj_n_sdsubjets",
    "fj_tau1", "fj_tau2", "fj_tau3",
    "fj_ptDR", "fj_relptdiff", "fj_jetNTracks",
    "fj_nSV", "fj_sdsj1_pt", "fj_sdsj1_mult",
    "fj_sdsj1_ptD", "fj_mass", "fj_eta", "fj_phi",
    "fj_sdsj1_eta", "fj_sdsj1_mass",
    "label_H_bb"
]

file = uproot.open(TRAIN_ROOT)
tree = file[TREE_NAME]

df_train = pd.DataFrame(tree.arrays(feature_cols, library="np"))

# derived features (same as before)
df_train["tau21"] = df_train["fj_tau2"] / df_train["fj_tau1"]
df_train["tau32"] = df_train["fj_tau3"] / df_train["fj_tau2"]

df_train.replace([float("inf"), -float("inf")], pd.NA, inplace=True)
df_train.fillna(df_train.median(numeric_only=True), inplace=True)

df_train.to_parquet(r"C:\Users\Agastya\Hbb-jet-tagging\data\processed\train.parquet", index=False)

print("✅ train.parquet recreated")


✅ train.parquet recreated
